In [ ]:
!nvidia-smi

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -q timm huggingface_hub safetensors pillow numpy pandas

In [ ]:
from pathlib import Path

from google.colab import userdata
from huggingface_hub import HfApi, login

raw_token = userdata.get("HF_TOKEN")

if not raw_token:
    raise RuntimeError(
        "HF_TOKEN is unavailable in Colab Secrets."
    )

token_lines = [
    line.strip()
    for line in raw_token.splitlines()
    if line.strip()
]

if len(token_lines) != 1:
    raise RuntimeError(
        "HF_TOKEN must contain exactly one non-empty line."
    )

HF_TOKEN = token_lines[0]

if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "HF_TOKEN has an unexpected format."
    )

if any(character.isspace() for character in HF_TOKEN):
    raise RuntimeError(
        "HF_TOKEN contains whitespace."
    )

login(
    token=HF_TOKEN,
    add_to_git_credential=False,
)

api = HfApi(token=HF_TOKEN)

identity = api.whoami()

DATASET_REPO = "AliothMe/lung-fusion-tile-shards"
DATASET_REVISION = (
    "899e7b2c754b8154887f4e818b9482dfbda0c9bc"
)

OUTPUT_REPO = "AliothMe/lung-fusion-embeddings"

VIRCHOW2_REPO = "paige-ai/Virchow2"

WORK_DIR = Path("/content/lung-fusion-agent")
INPUT_DIR = WORK_DIR / "input_shards"
OUTPUT_DIR = WORK_DIR / "virchow2_outputs"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Authenticated user:", identity["name"])
print("Input dataset:", DATASET_REPO)
print("Output dataset:", OUTPUT_REPO)
print("Model:", VIRCHOW2_REPO)

In [ ]:
from pathlib import Path

from google.colab import userdata
from huggingface_hub import HfApi, login

raw_token = userdata.get("HF_TOKEN")

if not raw_token:
    raise RuntimeError(
        "HF_TOKEN is unavailable in Colab Secrets."
    )

token_lines = [
    line.strip()
    for line in raw_token.splitlines()
    if line.strip()
]

if len(token_lines) != 1:
    raise RuntimeError(
        "HF_TOKEN must contain exactly one non-empty line."
    )

HF_TOKEN = token_lines[0]

if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "HF_TOKEN has an unexpected format."
    )

if any(character.isspace() for character in HF_TOKEN):
    raise RuntimeError(
        "HF_TOKEN contains whitespace."
    )

login(
    token=HF_TOKEN,
    add_to_git_credential=False,
)

api = HfApi(token=HF_TOKEN)

identity = api.whoami()

DATASET_REPO = "AliothMe/lung-fusion-tile-shards"
DATASET_REVISION = (
    "899e7b2c754b8154887f4e818b9482dfbda0c9bc"
)

OUTPUT_REPO = "AliothMe/lung-fusion-embeddings"

VIRCHOW2_REPO = "paige-ai/Virchow2"

WORK_DIR = Path("/content/lung-fusion-agent")
INPUT_DIR = WORK_DIR / "input_shards"
OUTPUT_DIR = WORK_DIR / "virchow2_outputs"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Authenticated user:", identity["name"])
print("Input dataset:", DATASET_REPO)
print("Output dataset:", OUTPUT_REPO)
print("Model:", VIRCHOW2_REPO)

In [ ]:
virchow2_info = api.model_info(
    VIRCHOW2_REPO
)

VIRCHOW2_REVISION = virchow2_info.sha

print(
    "Virchow2 revision:",
    VIRCHOW2_REVISION,
)

In [ ]:
import gc

import timm
import torch
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

VIRCHOW2_REVISION = (
    "3158645804b69e3f3bc4439d4116edddf0840a72"
)

device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

pinned_model_name = (
    f"hf-hub:{VIRCHOW2_REPO}@"
    f"{VIRCHOW2_REVISION}"
)

print("Loading:", pinned_model_name)

model = timm.create_model(
    pinned_model_name,
    pretrained=True,
    mlp_layer=timm.layers.SwiGLUPacked,
    act_layer=torch.nn.SiLU,
)

transform = create_transform(
    **resolve_data_config(
        model.pretrained_cfg,
        model=model,
    )
)

model = model.eval().to(device)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Model loaded.")
print("Parameters:", f"{parameter_count:,}")
print("Device:", next(model.parameters()).device)
print("Pretrained configuration:")
print(model.pretrained_cfg)

In [ ]:
from huggingface_hub import hf_hub_download

SHARD_FILENAME = "tiles-00000.tar"

shard_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename=SHARD_FILENAME,
    repo_type="dataset",
    revision=DATASET_REVISION,
    token=HF_TOKEN,
    local_dir=INPUT_DIR,
)

print("Downloaded:", shard_path)
print(
    "Size:",
    round(
        Path(shard_path).stat().st_size
        / 1024**2,
        2,
    ),
    "MiB",
)

In [ ]:
import io
import tarfile

from PIL import Image

SMOKE_TILE_COUNT = 128

with tarfile.open(shard_path, mode="r") as archive:
    jpg_members = [
        member
        for member in archive.getmembers()
        if member.isfile()
        and member.name.endswith(".jpg")
    ]

    selected_members = jpg_members[
        :SMOKE_TILE_COUNT
    ]

    tile_ids = []
    image_tensors = []

    for member in selected_members:
        extracted_file = archive.extractfile(member)

        if extracted_file is None:
            raise RuntimeError(
                f"Could not extract {member.name}"
            )

        image_bytes = extracted_file.read()

        with Image.open(
            io.BytesIO(image_bytes)
        ) as image:
            image = image.convert("RGB")
            image_tensor = transform(image)

        tile_ids.append(Path(member.name).stem)
        image_tensors.append(image_tensor)

images = torch.stack(image_tensors)

print("Available JPEG tiles:", len(jpg_members))
print("Loaded smoke tiles:", len(tile_ids))
print("Input tensor shape:", tuple(images.shape))
print("Input dtype:", images.dtype)
print("First tile:", tile_ids[0])

In [ ]:
import time

BATCH_SIZE = 32
embedding_batches = []

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.inference_mode():
    for start in range(
        0,
        len(images),
        BATCH_SIZE,
    ):
        batch = images[
            start : start + BATCH_SIZE
        ].to(
            device,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            token_output = model(batch)

        if token_output.shape[1:] != (261, 1280):
            raise RuntimeError(
                "Unexpected Virchow2 token shape: "
                f"{tuple(token_output.shape)}"
            )

        class_token = token_output[:, 0]
        patch_tokens = token_output[:, 5:]

        batch_embeddings = torch.cat(
            [
                class_token,
                patch_tokens.mean(dim=1),
            ],
            dim=-1,
        )

        embedding_batches.append(
            batch_embeddings.float().cpu()
        )

torch.cuda.synchronize()
elapsed_seconds = time.perf_counter() - start_time

embeddings = torch.cat(
    embedding_batches,
    dim=0,
)

peak_memory_gib = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("Final embedding shape:", tuple(embeddings.shape))
print("Embedding dtype:", embeddings.dtype)
print(
    "All finite:",
    bool(torch.isfinite(embeddings).all()),
)
print(
    "Elapsed seconds:",
    round(elapsed_seconds, 3),
)
print(
    "Tiles per second:",
    round(
        len(embeddings) / elapsed_seconds,
        2,
    ),
)
print(
    "Peak allocated GPU memory:",
    round(peak_memory_gib, 2),
    "GiB",
)
print(
    "Mean embedding L2 norm:",
    round(
        float(
            embeddings.norm(dim=1).mean()
        ),
        4,
    ),
)

In [ ]:
print(
    "Raw token output:",
    tuple(token_output.shape),
)
print(
    "Class token:",
    tuple(class_token.shape),
)
print(
    "Patch tokens:",
    tuple(patch_tokens.shape),
)
print(
    "Final batch embeddings:",
    tuple(batch_embeddings.shape),
)

In [ ]:
import statistics
import time

import torch


def make_virchow2_embeddings(
    token_output: torch.Tensor,
) -> torch.Tensor:
    if token_output.shape[1:] != (261, 1280):
        raise RuntimeError(
            "Unexpected Virchow2 token shape: "
            f"{tuple(token_output.shape)}"
        )

    class_token = token_output[:, 0]
    patch_tokens = token_output[:, 5:]

    return torch.cat(
        [
            class_token,
            patch_tokens.mean(dim=1),
        ],
        dim=-1,
    )


def benchmark_batch_size(
    batch_size: int,
    repeats: int = 3,
) -> dict[str, float]:
    sample_images = images[:128]
    times = []

    warmup_batch = sample_images[
        :batch_size
    ].to(
        device,
        non_blocking=True,
    )

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            warmup_tokens = model(warmup_batch)
            warmup_embeddings = (
                make_virchow2_embeddings(
                    warmup_tokens
                )
            )

    torch.cuda.synchronize()

    del warmup_batch
    del warmup_tokens
    del warmup_embeddings

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for _ in range(repeats):
        torch.cuda.synchronize()
        start_time = time.perf_counter()

        with torch.inference_mode():
            for start in range(
                0,
                len(sample_images),
                batch_size,
            ):
                batch = sample_images[
                    start : start + batch_size
                ].to(
                    device,
                    non_blocking=True,
                )

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.bfloat16,
                ):
                    token_output = model(batch)
                    batch_embeddings = (
                        make_virchow2_embeddings(
                            token_output
                        )
                    )

                del batch
                del token_output
                del batch_embeddings

        torch.cuda.synchronize()
        times.append(
            time.perf_counter() - start_time
        )

    median_seconds = statistics.median(times)
    tiles_per_second = (
        len(sample_images) / median_seconds
    )
    peak_memory_gib = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    return {
        "batch_size": batch_size,
        "median_seconds": median_seconds,
        "tiles_per_second": tiles_per_second,
        "peak_memory_gib": peak_memory_gib,
    }


benchmark_results = []

for tested_batch_size in [32, 64, 128]:
    result = benchmark_batch_size(
        tested_batch_size
    )
    benchmark_results.append(result)

    print(
        f"Batch {tested_batch_size:3d} | "
        f"{result['tiles_per_second']:7.2f} "
        "tiles/s | "
        f"{result['peak_memory_gib']:6.2f} GiB"
    )

In [ ]:
parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Parameters:", f"{parameter_count:,}")
print(
    "Parameters in millions:",
    round(parameter_count / 1_000_000, 2),
)

In [ ]:
import io
import tarfile
import time
from pathlib import Path

import numpy as np
from PIL import Image


BATCH_SIZE = 32
EXPECTED_TOKEN_SHAPE = (261, 1280)
EXPECTED_EMBEDDING_DIM = 2560


def infer_virchow2_batch(
    image_batch: list[torch.Tensor],
) -> np.ndarray:
    batch = torch.stack(image_batch).to(
        device,
        non_blocking=True,
    )

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            token_output = model(batch)

    if token_output.shape[1:] != EXPECTED_TOKEN_SHAPE:
        raise RuntimeError(
            "Unexpected Virchow2 token shape: "
            f"{tuple(token_output.shape)}"
        )

    class_token = token_output[:, 0]
    patch_tokens = token_output[:, 5:]

    batch_embeddings = torch.cat(
        [
            class_token,
            patch_tokens.mean(dim=1),
        ],
        dim=-1,
    )

    return (
        batch_embeddings
        .to(dtype=torch.float16)
        .cpu()
        .numpy()
    )


def extract_virchow2_shard(
    shard_path: str | Path,
) -> tuple[list[str], np.ndarray, float]:
    tile_ids: list[str] = []
    embedding_batches: list[np.ndarray] = []
    image_batch: list[torch.Tensor] = []

    torch.cuda.reset_peak_memory_stats()
    start_time = time.perf_counter()

    with tarfile.open(shard_path, mode="r") as archive:
        jpg_members = [
            member
            for member in archive.getmembers()
            if member.isfile()
            and member.name.endswith(".jpg")
        ]

        print("JPEG tiles found:", len(jpg_members))

        for tile_number, member in enumerate(
            jpg_members,
            start=1,
        ):
            extracted_file = archive.extractfile(member)

            if extracted_file is None:
                raise RuntimeError(
                    f"Could not extract {member.name}"
                )

            image_bytes = extracted_file.read()

            with Image.open(io.BytesIO(image_bytes)) as image:
                image = image.convert("RGB")
                image_tensor = transform(image)

            tile_ids.append(Path(member.name).stem)
            image_batch.append(image_tensor)

            if len(image_batch) == BATCH_SIZE:
                embedding_batches.append(
                    infer_virchow2_batch(
                        image_batch
                    )
                )
                image_batch.clear()

            if tile_number % 1024 == 0:
                print(
                    f"Processed {tile_number}/"
                    f"{len(jpg_members)} tiles"
                )

        if image_batch:
            embedding_batches.append(
                infer_virchow2_batch(
                    image_batch
                )
            )
            image_batch.clear()

    torch.cuda.synchronize()
    elapsed_seconds = (
        time.perf_counter() - start_time
    )

    embeddings = np.concatenate(
        embedding_batches,
        axis=0,
    )

    expected_shape = (
        len(tile_ids),
        EXPECTED_EMBEDDING_DIM,
    )

    if embeddings.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected shape {embeddings.shape}; "
            f"expected {expected_shape}"
        )

    if embeddings.dtype != np.float16:
        raise RuntimeError(
            f"Unexpected dtype {embeddings.dtype}"
        )

    if not np.isfinite(embeddings).all():
        raise RuntimeError(
            "Embeddings contain NaN or infinity."
        )

    if len(set(tile_ids)) != len(tile_ids):
        raise RuntimeError(
            "Duplicate tile IDs found."
        )

    return tile_ids, embeddings, elapsed_seconds

In [ ]:
import time

from huggingface_hub import HfApi
from huggingface_hub.errors import HfHubHTTPError


def robust_upload_file(*args, **kwargs):
    retry_delays = [0, 10, 20, 40]

    for attempt_number, delay_seconds in enumerate(
        retry_delays,
        start=1,
    ):
        if delay_seconds:
            print(
                f"Waiting {delay_seconds} seconds "
                "before retry..."
            )
            time.sleep(delay_seconds)

        try:
            result = HfApi.upload_file(
                api,
                *args,
                **kwargs,
            )

            print(
                f"Upload succeeded on attempt "
                f"{attempt_number}."
            )
            return result

        except HfHubHTTPError as error:
            print(
                f"Upload attempt {attempt_number} "
                f"failed: {type(error).__name__}"
            )

            if attempt_number == len(retry_delays):
                raise

    raise RuntimeError("Upload failed.")


api.upload_file = robust_upload_file

print("Upload retry wrapper installed.")

In [ ]:
import hashlib
import json

from huggingface_hub import hf_hub_download


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()


def process_and_upload_virchow2_shard(
    shard_index: int,
) -> dict[str, object]:
    input_name = f"tiles-{shard_index:05d}.tar"
    output_name = (
        f"virchow2-{shard_index:05d}.npz"
    )
    metadata_name = (
        f"virchow2-{shard_index:05d}.json"
    )

    remote_output_path = (
        f"virchow2/{output_name}"
    )
    remote_metadata_path = (
        f"virchow2/{metadata_name}"
    )

    print("=" * 60)
    print("Input:", input_name)
    print("Output:", remote_output_path)

    input_path = Path(
        hf_hub_download(
            repo_id=DATASET_REPO,
            filename=input_name,
            repo_type="dataset",
            revision=DATASET_REVISION,
            token=HF_TOKEN,
            local_dir=INPUT_DIR,
        )
    )

    print("Downloaded:", input_path)

    tile_ids, embeddings, elapsed_seconds = (
        extract_virchow2_shard(input_path)
    )

    output_path = OUTPUT_DIR / output_name
    metadata_path = OUTPUT_DIR / metadata_name

    np.savez_compressed(
        output_path,
        tile_ids=np.asarray(tile_ids),
        embeddings=embeddings,
    )

    output_checksum = sha256_file(output_path)
    peak_memory_gib = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    metadata = {
        "model_name": "Virchow2",
        "model_repo": VIRCHOW2_REPO,
        "model_revision": VIRCHOW2_REVISION,
        "dataset_repo": DATASET_REPO,
        "dataset_revision": DATASET_REVISION,
        "input_shard": input_name,
        "output_file": output_name,
        "tile_count": len(tile_ids),
        "raw_token_count": 261,
        "raw_token_dimension": 1280,
        "register_tokens_excluded": 4,
        "embedding_aggregation": (
            "concat_cls_and_mean_patch_tokens"
        ),
        "embedding_dimension": int(
            embeddings.shape[1]
        ),
        "embedding_dtype": str(embeddings.dtype),
        "batch_size": BATCH_SIZE,
        "elapsed_seconds": elapsed_seconds,
        "tiles_per_second": (
            len(tile_ids) / elapsed_seconds
        ),
        "peak_gpu_memory_gib": peak_memory_gib,
        "output_size_bytes": (
            output_path.stat().st_size
        ),
        "output_sha256": output_checksum,
        "gpu": torch.cuda.get_device_name(0),
        "torch_version": torch.__version__,
        "timm_version": timm.__version__,
    }

    metadata_path.write_text(
        json.dumps(
            metadata,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    print("Local validation:")
    print("  Shape:", embeddings.shape)
    print("  Dtype:", embeddings.dtype)
    print(
        "  Finite:",
        bool(np.isfinite(embeddings).all()),
    )
    print(
        "  Output size:",
        round(
            output_path.stat().st_size
            / 1024**2,
            2,
        ),
        "MiB",
    )

    api.upload_file(
        repo_id=OUTPUT_REPO,
        repo_type="dataset",
        path_or_fileobj=str(output_path),
        path_in_repo=remote_output_path,
        commit_message=(
            f"Add Virchow2 shard "
            f"{shard_index:05d}"
        ),
    )

    api.upload_file(
        repo_id=OUTPUT_REPO,
        repo_type="dataset",
        path_or_fileobj=str(metadata_path),
        path_in_repo=remote_metadata_path,
        commit_message=(
            f"Add Virchow2 metadata "
            f"{shard_index:05d}"
        ),
    )

    print(
        f"Completed {input_name}: "
        f"{len(tile_ids)} tiles, "
        f"{len(tile_ids) / elapsed_seconds:.2f} "
        "tiles/s"
    )

    return metadata

In [ ]:
required_names = [
    "HF_TOKEN",
    "api",
    "model",
    "transform",
    "DATASET_REPO",
    "DATASET_REVISION",
    "OUTPUT_REPO",
    "VIRCHOW2_REPO",
    "VIRCHOW2_REVISION",
    "INPUT_DIR",
    "OUTPUT_DIR",
    "process_and_upload_virchow2_shard",
]

for name in required_names:
    print(
        f"{name}:",
        "ready" if name in globals() else "MISSING",
    )

In [ ]:
pilot_metadata = (
    process_and_upload_virchow2_shard(0)
)

print("\n=== Virchow2 pilot result ===")
print(
    json.dumps(
        pilot_metadata,
        indent=2,
        sort_keys=True,
    )
)

In [ ]:
remote_path = hf_hub_download(
    repo_id=OUTPUT_REPO,
    filename=(
        "virchow2/virchow2-00000.npz"
    ),
    repo_type="dataset",
    token=HF_TOKEN,
    force_download=True,
)

with np.load(
    remote_path,
    allow_pickle=False,
) as remote_cache:
    remote_ids = remote_cache["tile_ids"]
    remote_embeddings = (
        remote_cache["embeddings"]
    )

    print("Tile IDs:", remote_ids.shape)
    print(
        "Embeddings:",
        remote_embeddings.shape,
    )
    print("Dtype:", remote_embeddings.dtype)
    print(
        "All finite:",
        bool(
            np.isfinite(
                remote_embeddings
            ).all()
        ),
    )
    print(
        "Unique tile IDs:",
        len(set(remote_ids.tolist())),
    )
    print(
        "Mean L2 norm:",
        round(
            float(
                np.linalg.norm(
                    remote_embeddings.astype(
                        np.float32
                    ),
                    axis=1,
                ).mean()
            ),
            4,
        ),
    )

In [ ]:
import time

TOTAL_SHARDS = 49
RUN_START_SHARD = 0
RUN_END_SHARD = 49

remote_files = set(
    api.list_repo_files(
        OUTPUT_REPO,
        repo_type="dataset",
    )
)

completed_this_run = []
skipped_this_run = []

full_run_start = time.perf_counter()

for shard_index in range(
    RUN_START_SHARD,
    RUN_END_SHARD,
):
    embedding_path = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.npz"
    )
    metadata_path = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.json"
    )

    embedding_exists = (
        embedding_path in remote_files
    )
    metadata_exists = (
        metadata_path in remote_files
    )

    print("\n" + "=" * 70)
    print(
        f"Shard {shard_index + 1}/"
        f"{TOTAL_SHARDS}: "
        f"{shard_index:05d}"
    )

    if embedding_exists and metadata_exists:
        print(
            "Remote outputs already exist; "
            "skipping."
        )
        skipped_this_run.append(shard_index)
        continue

    if embedding_exists != metadata_exists:
        print(
            "Partial remote output detected. "
            "The shard will be regenerated."
        )

    shard_metadata = (
        process_and_upload_virchow2_shard(
            shard_index
        )
    )

    remote_files.add(embedding_path)
    remote_files.add(metadata_path)
    completed_this_run.append(
        shard_metadata
    )

    elapsed_total = (
        time.perf_counter() - full_run_start
    )

    completed_count = len(
        completed_this_run
    )
    handled_count = (
        completed_count
        + len(skipped_this_run)
    )

    average_seconds = (
        elapsed_total
        / max(completed_count, 1)
    )

    remaining_count = (
        RUN_END_SHARD
        - shard_index
        - 1
    )

    estimated_remaining_minutes = (
        average_seconds
        * remaining_count
        / 60
    )

    print(
        f"Progress: {handled_count}/"
        f"{RUN_END_SHARD - RUN_START_SHARD}"
    )
    print(
        "Estimated remaining time:",
        round(
            estimated_remaining_minutes,
            1,
        ),
        "minutes",
    )

full_run_elapsed = (
    time.perf_counter() - full_run_start
)

print("\n=== Virchow2 full run finished ===")
print(
    "Newly completed shards:",
    len(completed_this_run),
)
print(
    "Skipped existing shards:",
    len(skipped_this_run),
)
print(
    "Total elapsed minutes:",
    round(full_run_elapsed / 60, 2),
)

In [ ]:
failed_npz = (
    OUTPUT_DIR / "virchow2-00013.npz"
)
failed_json = (
    OUTPUT_DIR / "virchow2-00013.json"
)

print("NPZ exists:", failed_npz.exists())
print("JSON exists:", failed_json.exists())

if failed_npz.exists():
    print(
        "NPZ size:",
        round(
            failed_npz.stat().st_size
            / 1024**2,
            2,
        ),
        "MiB",
    )

In [ ]:
from huggingface_hub import HfApi

upload_api = HfApi(token=HF_TOKEN)

print("Batch upload client ready.")

In [ ]:
def defer_upload(*args, **kwargs):
    path_in_repo = kwargs.get(
        "path_in_repo",
        "[unknown]",
    )

    print(
        "Upload deferred for batch commit:",
        path_in_repo,
    )

    return None


api.upload_file = defer_upload

print("Per-file uploads disabled.")

In [ ]:
import json
import time

TOTAL_SHARDS = 49

remote_files = set(
    upload_api.list_repo_files(
        OUTPUT_REPO,
        repo_type="dataset",
    )
)

remote_complete = []
local_pending = []
newly_computed = []

local_run_start = time.perf_counter()

for shard_index in range(TOTAL_SHARDS):
    remote_npz = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.npz"
    )
    remote_json = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.json"
    )

    local_npz = (
        OUTPUT_DIR
        / f"virchow2-{shard_index:05d}.npz"
    )
    local_json = (
        OUTPUT_DIR
        / f"virchow2-{shard_index:05d}.json"
    )

    remote_has_both = (
        remote_npz in remote_files
        and remote_json in remote_files
    )

    local_has_both = (
        local_npz.exists()
        and local_json.exists()
    )

    print("\n" + "=" * 70)
    print(
        f"Shard {shard_index + 1}/49: "
        f"{shard_index:05d}"
    )

    if remote_has_both:
        print("Complete on remote; skipping.")
        remote_complete.append(shard_index)
        continue

    if local_has_both:
        print(
            "Complete locally; waiting for "
            "batch upload."
        )
        local_pending.append(shard_index)
        continue

    print("Not complete; computing locally.")

    metadata = (
        process_and_upload_virchow2_shard(
            shard_index
        )
    )

    if not local_npz.exists():
        raise RuntimeError(
            f"Missing local NPZ after processing "
            f"shard {shard_index}"
        )

    if not local_json.exists():
        raise RuntimeError(
            f"Missing local JSON after processing "
            f"shard {shard_index}"
        )

    newly_computed.append(shard_index)
    local_pending.append(shard_index)

elapsed_minutes = (
    time.perf_counter() - local_run_start
) / 60

print("\n=== Local extraction complete ===")
print(
    "Already complete remotely:",
    len(remote_complete),
)
print(
    "Already complete locally:",
    len(local_pending)
    - len(newly_computed),
)
print(
    "Newly computed:",
    len(newly_computed),
)
print(
    "Pending batch upload:",
    len(local_pending),
)
print(
    "Elapsed minutes:",
    round(elapsed_minutes, 2),
)

In [ ]:
local_npz_files = sorted(
    OUTPUT_DIR.glob("virchow2-*.npz")
)

local_json_files = sorted(
    OUTPUT_DIR.glob("virchow2-*.json")
)

print("Local NPZ files:", len(local_npz_files))
print("Local JSON files:", len(local_json_files))
print(
    "Total local size:",
    round(
        sum(
            path.stat().st_size
            for path in local_npz_files
        )
        / 1024**3,
        3,
    ),
    "GiB",
)

In [ ]:
upload_result = upload_api.upload_folder(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    folder_path=str(OUTPUT_DIR),
    path_in_repo="virchow2",
    commit_message=(
        "Upload complete Virchow2 "
        "embedding cache"
    ),
)

print("Batch upload complete:")
print(upload_result)

In [ ]:
remote_files = upload_api.list_repo_files(
    OUTPUT_REPO,
    repo_type="dataset",
)

remote_npz_files = sorted(
    path
    for path in remote_files
    if path.startswith("virchow2/")
    and path.endswith(".npz")
)

remote_json_files = sorted(
    path
    for path in remote_files
    if path.startswith("virchow2/")
    and path.endswith(".json")
)

print(
    "Remote Virchow2 NPZ files:",
    len(remote_npz_files),
)
print(
    "Remote Virchow2 JSON files:",
    len(remote_json_files),
)

In [ ]:
import json

import pandas as pd
from huggingface_hub import hf_hub_download

metadata_records = []

for shard_index in range(49):
    metadata_filename = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.json"
    )

    local_metadata_path = hf_hub_download(
        repo_id=OUTPUT_REPO,
        filename=metadata_filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    with open(
        local_metadata_path,
        encoding="utf-8",
    ) as metadata_file:
        metadata_records.append(
            json.load(metadata_file)
        )

virchow2_manifest = pd.DataFrame(
    metadata_records
).sort_values(
    "input_shard"
).reset_index(drop=True)

print("=== Virchow2 metadata summary ===")
print("Records:", len(virchow2_manifest))
print(
    "Total tiles:",
    int(
        virchow2_manifest[
            "tile_count"
        ].sum()
    ),
)
print(
    "Embedding dimensions:",
    virchow2_manifest[
        "embedding_dimension"
    ].unique(),
)
print(
    "Embedding dtypes:",
    virchow2_manifest[
        "embedding_dtype"
    ].unique(),
)
print(
    "Batch sizes:",
    virchow2_manifest[
        "batch_size"
    ].unique(),
)
print(
    "Model revisions:",
    virchow2_manifest[
        "model_revision"
    ].unique(),
)
print(
    "Dataset revisions:",
    virchow2_manifest[
        "dataset_revision"
    ].unique(),
)
print(
    "Aggregation:",
    virchow2_manifest[
        "embedding_aggregation"
    ].unique(),
)
print(
    "Total cached size:",
    round(
        virchow2_manifest[
            "output_size_bytes"
        ].sum()
        / 1024**3,
        3,
    ),
    "GiB",
)
print(
    "Mean end-to-end speed:",
    round(
        virchow2_manifest[
            "tiles_per_second"
        ].mean(),
        2,
    ),
    "tiles/s",
)
print(
    "Min speed:",
    round(
        virchow2_manifest[
            "tiles_per_second"
        ].min(),
        2,
    ),
)
print(
    "Max speed:",
    round(
        virchow2_manifest[
            "tiles_per_second"
        ].max(),
        2,
    ),
)

In [ ]:
virchow2_manifest_path = (
    OUTPUT_DIR / "virchow2_manifest.csv"
)

virchow2_manifest.to_csv(
    virchow2_manifest_path,
    index=False,
)

print(
    "Saved:",
    virchow2_manifest_path,
)

In [ ]:
upload_api.upload_file(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    path_or_fileobj=str(
        virchow2_manifest_path
    ),
    path_in_repo="virchow2/manifest.csv",
    commit_message=(
        "Add Virchow2 extraction manifest"
    ),
)

print(
    "Uploaded: virchow2/manifest.csv"
)

In [ ]:
embedding_repo_info = upload_api.repo_info(
    OUTPUT_REPO,
    repo_type="dataset",
)

VIRCHOW2_CACHE_REVISION = (
    embedding_repo_info.sha
)

print(
    "Virchow2 cache revision:",
    VIRCHOW2_CACHE_REVISION,
)

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

tile_index_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename="manifests/tile_to_shard.csv.gz",
    repo_type="dataset",
    revision=DATASET_REVISION,
    token=HF_TOKEN,
)

expected_tile_index = pd.read_csv(
    tile_index_path
)

print("Expected tiles:", len(expected_tile_index))
print(
    "Unique tile IDs:",
    expected_tile_index["tile_id"].nunique(),
)
print(
    "Unique shards:",
    expected_tile_index[
        "shard_name"
    ].nunique(),
)

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np


VIRCHOW2_CACHE_REVISION = (
    "cb5a359a317ad2411ba1c4de78c60816772e04a1"
)


def calculate_sha256(path: str | Path) -> str:
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        while chunk := file.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()


validation_records = []
all_observed_tile_ids = set()

total_embedding_rows = 0
total_norm_sum = 0.0
global_norm_min = float("inf")
global_norm_max = float("-inf")

for shard_index in range(49):
    input_shard_name = (
        f"tiles-{shard_index:05d}.tar"
    )
    embedding_filename = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.npz"
    )
    metadata_filename = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.json"
    )

    expected_frame = expected_tile_index.loc[
        expected_tile_index["shard_name"]
        == input_shard_name
    ]

    expected_ids = (
        expected_frame["tile_id"]
        .astype(str)
        .to_numpy()
    )

    embedding_path = hf_hub_download(
        repo_id=OUTPUT_REPO,
        filename=embedding_filename,
        repo_type="dataset",
        revision=VIRCHOW2_CACHE_REVISION,
        token=HF_TOKEN,
    )

    metadata_path = hf_hub_download(
        repo_id=OUTPUT_REPO,
        filename=metadata_filename,
        repo_type="dataset",
        revision=VIRCHOW2_CACHE_REVISION,
        token=HF_TOKEN,
    )

    with open(
        metadata_path,
        encoding="utf-8",
    ) as metadata_file:
        shard_metadata = json.load(
            metadata_file
        )

    with np.load(
        embedding_path,
        allow_pickle=False,
    ) as cache:
        observed_ids = (
            cache["tile_ids"]
            .astype(str)
        )
        observed_embeddings = (
            cache["embeddings"]
        )

        expected_shape = (
            len(expected_ids),
            2560,
        )

        if observed_embeddings.shape != expected_shape:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"shape {observed_embeddings.shape}, "
                f"expected {expected_shape}"
            )

        if observed_embeddings.dtype != np.float16:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"unexpected dtype "
                f"{observed_embeddings.dtype}"
            )

        if not np.isfinite(
            observed_embeddings
        ).all():
            raise RuntimeError(
                f"{embedding_filename}: "
                "contains NaN or infinity"
            )

        if len(np.unique(observed_ids)) != len(
            observed_ids
        ):
            raise RuntimeError(
                f"{embedding_filename}: "
                "duplicate IDs within shard"
            )

        if not np.array_equal(
            observed_ids,
            expected_ids,
        ):
            observed_set = set(
                observed_ids.tolist()
            )
            expected_set = set(
                expected_ids.tolist()
            )

            missing = expected_set - observed_set
            extra = observed_set - expected_set

            raise RuntimeError(
                f"{embedding_filename}: "
                "tile IDs do not match; "
                f"missing={len(missing)}, "
                f"extra={len(extra)}"
            )

        observed_id_set = set(
            observed_ids.tolist()
        )
        overlap = (
            all_observed_tile_ids
            & observed_id_set
        )

        if overlap:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"{len(overlap)} IDs already "
                "appeared in another shard"
            )

        all_observed_tile_ids.update(
            observed_id_set
        )

        embedding_float32 = (
            observed_embeddings.astype(
                np.float32
            )
        )

        norms = np.linalg.norm(
            embedding_float32,
            axis=1,
        )

        total_embedding_rows += len(
            observed_embeddings
        )
        total_norm_sum += float(norms.sum())
        global_norm_min = min(
            global_norm_min,
            float(norms.min()),
        )
        global_norm_max = max(
            global_norm_max,
            float(norms.max()),
        )

        shard_mean_norm = float(
            norms.mean()
        )
        shard_min_norm = float(
            norms.min()
        )
        shard_max_norm = float(
            norms.max()
        )

    observed_checksum = calculate_sha256(
        embedding_path
    )
    expected_checksum = shard_metadata[
        "output_sha256"
    ]

    if observed_checksum != expected_checksum:
        raise RuntimeError(
            f"{embedding_filename}: "
            "SHA-256 mismatch"
        )

    validation_records.append(
        {
            "shard_index": shard_index,
            "input_shard": input_shard_name,
            "embedding_file": (
                embedding_filename
            ),
            "tile_count": len(expected_ids),
            "embedding_dimension": 2560,
            "embedding_dtype": "float16",
            "all_finite": True,
            "tile_order_matches": True,
            "sha256_matches": True,
            "mean_l2_norm": shard_mean_norm,
            "min_l2_norm": shard_min_norm,
            "max_l2_norm": shard_max_norm,
        }
    )

    print(
        f"[{shard_index + 1:02d}/49] "
        f"{embedding_filename}: "
        f"{len(expected_ids)} tiles ✓"
    )

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np


VIRCHOW2_CACHE_REVISION = (
    "cb5a359a317ad2411ba1c4de78c60816772e04a1"
)


def calculate_sha256(path: str | Path) -> str:
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        while chunk := file.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()


validation_records = []
all_observed_tile_ids = set()

total_embedding_rows = 0
total_norm_sum = 0.0
global_norm_min = float("inf")
global_norm_max = float("-inf")

for shard_index in range(49):
    input_shard_name = (
        f"tiles-{shard_index:05d}.tar"
    )
    embedding_filename = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.npz"
    )
    metadata_filename = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.json"
    )

    expected_frame = expected_tile_index.loc[
        expected_tile_index["shard_name"]
        == input_shard_name
    ]

    expected_ids = (
        expected_frame["tile_id"]
        .astype(str)
        .to_numpy()
    )

    embedding_path = hf_hub_download(
        repo_id=OUTPUT_REPO,
        filename=embedding_filename,
        repo_type="dataset",
        revision=VIRCHOW2_CACHE_REVISION,
        token=HF_TOKEN,
    )

    metadata_path = hf_hub_download(
        repo_id=OUTPUT_REPO,
        filename=metadata_filename,
        repo_type="dataset",
        revision=VIRCHOW2_CACHE_REVISION,
        token=HF_TOKEN,
    )

    with open(
        metadata_path,
        encoding="utf-8",
    ) as metadata_file:
        shard_metadata = json.load(
            metadata_file
        )

    with np.load(
        embedding_path,
        allow_pickle=False,
    ) as cache:
        observed_ids = (
            cache["tile_ids"]
            .astype(str)
        )
        observed_embeddings = (
            cache["embeddings"]
        )

        expected_shape = (
            len(expected_ids),
            2560,
        )

        if observed_embeddings.shape != expected_shape:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"shape {observed_embeddings.shape}, "
                f"expected {expected_shape}"
            )

        if observed_embeddings.dtype != np.float16:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"unexpected dtype "
                f"{observed_embeddings.dtype}"
            )

        if not np.isfinite(
            observed_embeddings
        ).all():
            raise RuntimeError(
                f"{embedding_filename}: "
                "contains NaN or infinity"
            )

        if len(np.unique(observed_ids)) != len(
            observed_ids
        ):
            raise RuntimeError(
                f"{embedding_filename}: "
                "duplicate IDs within shard"
            )

        if not np.array_equal(
            observed_ids,
            expected_ids,
        ):
            observed_set = set(
                observed_ids.tolist()
            )
            expected_set = set(
                expected_ids.tolist()
            )

            missing = expected_set - observed_set
            extra = observed_set - expected_set

            raise RuntimeError(
                f"{embedding_filename}: "
                "tile IDs do not match; "
                f"missing={len(missing)}, "
                f"extra={len(extra)}"
            )

        observed_id_set = set(
            observed_ids.tolist()
        )
        overlap = (
            all_observed_tile_ids
            & observed_id_set
        )

        if overlap:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"{len(overlap)} IDs already "
                "appeared in another shard"
            )

        all_observed_tile_ids.update(
            observed_id_set
        )

        embedding_float32 = (
            observed_embeddings.astype(
                np.float32
            )
        )

        norms = np.linalg.norm(
            embedding_float32,
            axis=1,
        )

        total_embedding_rows += len(
            observed_embeddings
        )
        total_norm_sum += float(norms.sum())
        global_norm_min = min(
            global_norm_min,
            float(norms.min()),
        )
        global_norm_max = max(
            global_norm_max,
            float(norms.max()),
        )

        shard_mean_norm = float(
            norms.mean()
        )
        shard_min_norm = float(
            norms.min()
        )
        shard_max_norm = float(
            norms.max()
        )

    observed_checksum = calculate_sha256(
        embedding_path
    )
    expected_checksum = shard_metadata[
        "output_sha256"
    ]

    if observed_checksum != expected_checksum:
        raise RuntimeError(
            f"{embedding_filename}: "
            "SHA-256 mismatch"
        )

    validation_records.append(
        {
            "shard_index": shard_index,
            "input_shard": input_shard_name,
            "embedding_file": (
                embedding_filename
            ),
            "tile_count": len(expected_ids),
            "embedding_dimension": 2560,
            "embedding_dtype": "float16",
            "all_finite": True,
            "tile_order_matches": True,
            "sha256_matches": True,
            "mean_l2_norm": shard_mean_norm,
            "min_l2_norm": shard_min_norm,
            "max_l2_norm": shard_max_norm,
        }
    )

    print(
        f"[{shard_index + 1:02d}/49] "
        f"{embedding_filename}: "
        f"{len(expected_ids)} tiles ✓"
    )

In [ ]:
expected_global_ids = set(
    expected_tile_index[
        "tile_id"
    ].astype(str)
)

missing_global_ids = (
    expected_global_ids
    - all_observed_tile_ids
)
extra_global_ids = (
    all_observed_tile_ids
    - expected_global_ids
)

if total_embedding_rows != 200488:
    raise RuntimeError(
        f"Expected 200488 rows, "
        f"found {total_embedding_rows}"
    )

if missing_global_ids:
    raise RuntimeError(
        f"Missing {len(missing_global_ids)} "
        "tile IDs"
    )

if extra_global_ids:
    raise RuntimeError(
        f"Found {len(extra_global_ids)} "
        "unexpected tile IDs"
    )

global_mean_norm = (
    total_norm_sum / total_embedding_rows
)

print("=== Virchow2 global validation ===")
print(
    "Embedding shards:",
    len(validation_records),
)
print(
    "Total embedding rows:",
    total_embedding_rows,
)
print(
    "Unique observed tile IDs:",
    len(all_observed_tile_ids),
)
print(
    "Missing tile IDs:",
    len(missing_global_ids),
)
print(
    "Extra tile IDs:",
    len(extra_global_ids),
)
print(
    "All dimensions:",
    sorted(
        {
            record["embedding_dimension"]
            for record in validation_records
        }
    ),
)
print(
    "All dtypes:",
    sorted(
        {
            record["embedding_dtype"]
            for record in validation_records
        }
    ),
)
print(
    "All finite:",
    all(
        record["all_finite"]
        for record in validation_records
    ),
)
print(
    "All tile orders match:",
    all(
        record["tile_order_matches"]
        for record in validation_records
    ),
)
print(
    "All checksums match:",
    all(
        record["sha256_matches"]
        for record in validation_records
    ),
)
print(
    "Global mean L2 norm:",
    round(global_mean_norm, 4),
)
print(
    "Global min L2 norm:",
    round(global_norm_min, 4),
)
print(
    "Global max L2 norm:",
    round(global_norm_max, 4),
)

In [ ]:
validation_frame = pd.DataFrame(
    validation_records
)

validation_output_path = (
    OUTPUT_DIR / "virchow2_validation.csv"
)

validation_frame.to_csv(
    validation_output_path,
    index=False,
)

upload_api.upload_file(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    path_or_fileobj=str(
        validation_output_path
    ),
    path_in_repo=(
        "virchow2/validation.csv"
    ),
    commit_message=(
        "Add Virchow2 validation report"
    ),
)

final_embedding_info = upload_api.repo_info(
    OUTPUT_REPO,
    repo_type="dataset",
)

print(
    "Validation report uploaded:",
    "virchow2/validation.csv",
)
print(
    "Final embedding revision:",
    final_embedding_info.sha,
)